# Module 5: Krum Defense


## Stage Goal

This notebook isolates Krum aggregation. It loads the saved FedAvg baselines from `fedavg_baselines.ipynb`, runs one attacked Krum defense, and compares it against clean and attacked FedAvg. Run `fedavg_baselines.ipynb` first so the clean and attacked FedAvg artifacts exist. Sweeps stay in `defense_sweeps.ipynb`.


## 1. Notebook Setup


In [ ]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "5_Defensive_FL").exists():
    MODULE_DIR = MODULE_DIR / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
for path in (MODULE_DIR.parent, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_text = str(path.resolve())
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from IPython.display import Image, display
import pandas as pd

from src.notebook_utils import (
    load_required_baselines,
    prepare_context,
    record_config_snapshot,
    run_module5_experiment,
    run_result_path,
    validate_artifacts,
    validate_result_collection,
)
from src.metrics import build_comparison_rows, latest_defense_diagnostics
from src.plots import (
    plot_accuracy_curves,
    plot_defense_comparison,
    plot_global_target_label_asr_curves,
    plot_surrogate_poison_success_curves,
)


## 2. Configuration

Edit this cell to change data, FL, attack, or defense settings. No YAML config is used.


In [ ]:
BASE_FED_CONFIG = {
    "algorithm": "FedAvg",
    "fraction_clients": 0.2,
    "num_clients": 50,
    "num_rounds": 12,
    "num_epochs": 3,
    "batch_size": 64,
    "global_stepsize": 1.0,
    "local_stepsize": 0.002,
    "criterion": "torch.nn.CrossEntropyLoss",
}

RUN_NAME = 'krum'
DEFENSE_TITLE = 'Krum'
DEFENSE_CONFIG = {'name': 'krum', 'byzantine_f': 2}

CONFIG = {
    "data_config": {
        "dataset_path": "./Data/Imagenette",
        "dataset_name": "Imagenette",
        "non_iid_per": 0,
        "eval_subset": "attack_eval",
        "validation_split": {"enabled": True, "seed": 42, "selection_fraction": 0.5},
    },
    "global_config": {"seed": 42, "device": "cuda"},
    "module4_handoff": {
        "enabled": True,
        "artifacts_dir": "../4_Adversarial_FL/artifacts",
        "target_checkpoint": "module4_v3_target.pt",
        "surrogate_checkpoint": "module4_surrogate.pt",
    },
    "artifacts": {
        "dir": "artifacts",
        "config_snapshot": f"module5_{RUN_NAME}_config_used.json",
        "clean_baseline_metrics": "module5_clean_fedavg.json",
        "attacked_baseline_metrics": "module5_attacked_fedavg.json",
    },
    "stage": {"name": f"{RUN_NAME}_defense", "notebook": 'krum_defense.ipynb'},
    "model_config": {
        "module": "model",
        "name": "MobileNetV3Transfer",
        "kwargs": {"pretrained": False, "num_classes": 10, "dropout": 0.1},
    },
    "algorithms": {
        "FedAvg": {
            "fed_config": BASE_FED_CONFIG,
            "optim_config": {},
        }
    },
    "defense": DEFENSE_CONFIG,
    "attack": {
        "seed": 42,
        "malicious_fraction": 0.1,
        "malicious_client_selection": {"mode": "seeded_random", "client_ids": []},
        "start_round": 3,
        "attack": {
            "type": "pgd",
            "poison_rate": 0.2,
            "target_label": 0,
            "epsilon": 8 / 255,
            "step_size": 2 / 255,
            "iters": 20,
            "criterion": "torch.nn.CrossEntropyLoss",
            "poison_rate_schedule": {"type": "linear", "start": 0.05, "end": 0.2},
        },
        "surrogate": {
            "checkpoint": "../4_Adversarial_FL/artifacts/module4_surrogate.pt",
            "checkpoint_source": "train_surrogate.ipynb",
            "pretrained": False,
            "finetune_epochs": 0,
            "local_finetune_epochs": 0,
            "learning_rate": 0.001,
            "weight_decay": 0.0,
            "batch_size": 64,
            "freeze_backbone": False,
            "early_stop_patience": 0,
        },
    },
}

context = prepare_context(CONFIG, stage_name=f"{RUN_NAME}_defense")
config_snapshot_path = record_config_snapshot(context)
print(f"Defense: {DEFENSE_TITLE}")
print(f"Sampled clients per round: {context.sampled_clients}")
print(f"Config snapshot: {config_snapshot_path}")


## 3. Load FedAvg Baseline Artifacts

This loads the clean and attacked FedAvg results saved by `fedavg_baselines.ipynb`; it does not rerun those baselines.


In [ ]:
baseline_results = load_required_baselines(context)
baseline_validation = validate_result_collection(
    context,
    baseline_results,
    required_runs=["clean_fedavg", "attacked_fedavg"],
)

display(pd.DataFrame(baseline_validation))


## 4. Run Krum

This runs one attacked FL job with only `DEFENSE_CONFIG` changed from attacked FedAvg.


In [ ]:
defense_result = run_module5_experiment(
    context,
    RUN_NAME,
    context.base_attack_config,
    DEFENSE_CONFIG,
)
run_results = {**baseline_results, RUN_NAME: defense_result}
comparison_rows = build_comparison_rows(run_results)
comparison_table = pd.DataFrame(comparison_rows)

display(comparison_table)
display(pd.DataFrame(validate_result_collection(context, {RUN_NAME: defense_result})))
latest_defense_diagnostics(defense_result)


## 5. Plots and Artifact Validation

The plots compare clean FedAvg, attacked FedAvg, and this defense notebook's run.


In [ ]:
plot_paths = [
    plot_accuracy_curves(
        run_results,
        context.artifact_path(f"module5_{RUN_NAME}_accuracy_curves.png"),
        attack_start_round=context.base_attack_config["start_round"],
        title=f"{DEFENSE_TITLE}: accuracy by round",
    ),
    plot_surrogate_poison_success_curves(
        run_results,
        context.artifact_path(f"module5_{RUN_NAME}_surrogate_poison_success_curves.png"),
        attack_start_round=context.base_attack_config["start_round"],
        title=f"{DEFENSE_TITLE}: surrogate poison success by round",
    ),
    plot_global_target_label_asr_curves(
        run_results,
        context.artifact_path(f"module5_{RUN_NAME}_global_target_label_asr_curves.png"),
        attack_start_round=context.base_attack_config["start_round"],
        title=f"{DEFENSE_TITLE}: global target-label ASR by round",
    ),
    plot_defense_comparison(
        comparison_rows,
        metric="final_accuracy",
        path=context.artifact_path(f"module5_{RUN_NAME}_accuracy_vs_baselines.png"),
        ylabel="Final accuracy (%)",
        title=f"{DEFENSE_TITLE}: final accuracy",
    ),
    plot_defense_comparison(
        comparison_rows,
        metric="final_surrogate_poison_success_rate",
        path=context.artifact_path(f"module5_{RUN_NAME}_surrogate_poison_success_vs_baselines.png"),
        ylabel="Final surrogate poison success rate (%)",
        title=f"{DEFENSE_TITLE}: final surrogate poison success",
    ),
    plot_defense_comparison(
        comparison_rows,
        metric="final_global_target_label_asr",
        path=context.artifact_path(f"module5_{RUN_NAME}_global_target_label_asr_vs_baselines.png"),
        ylabel="Final global target-label ASR (%)",
        title=f"{DEFENSE_TITLE}: final global target-label ASR",
    ),
]

validate_artifacts([config_snapshot_path, run_result_path(context, RUN_NAME), *plot_paths])
for plot_path in plot_paths:
    display(Image(filename=str(plot_path)))
